In [ ]:
#| export
# allos/io/junctions.py
# ============================================================
# Junction utilities (SiCeLoRe-style juncmatrix -> AnnData Zarr)
#
# 1) juncmatrix_to_zarr:       one juncmatrix(.txt|.gz) -> AnnData Zarr (cells×junctions)
# 2) batch_juncmatrix_to_zarr: many juncmatrix files (or a root/glob) -> per-sample Zarrs
# 3) merge_junction_zarrs:     merge per-sample junction Zarrs -> one cohort Zarr
#
# GUARANTEE:
#   - output AnnData.var ALWAYS contains 'geneId'
#   - var contains ONLY 'geneId' (no chr/start/end/strand)
#
# IMPORTANT:
#   Many SiCeLoRe junction IDs are of the form:
#       GENE_NAME:START-END
#   For that case, the *prefix itself* is a valid per-gene key for slicing.
#   If a GTF is provided, we TRY to map gene_name -> Ensembl gene_id, but we ALWAYS
#   fallback to the prefix so geneId is never NaN.
# ============================================================

from __future__ import annotations

import gzip
import re
import glob
from pathlib import Path
from typing import Iterable, Optional, Union, Sequence

import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm


def _is_gz(p: Union[str, Path]) -> bool:
    return str(p).endswith(".gz")


def _read_header_barcodes(junc_path: Path) -> list[str]:
    kw = dict(sep="\t", dtype=str, nrows=0)
    if _is_gz(junc_path):
        kw["compression"] = "gzip"
    hdr = pd.read_csv(junc_path, **kw)
    if hdr.shape[1] < 2:
        raise ValueError("Expected first col = junction id, then at least 1 barcode column.")
    return list(hdr.columns[1:])


def _stream_chunks(junc_path: Path, chunksize: int):
    kw = dict(sep="\t", dtype=str, chunksize=chunksize)
    if _is_gz(junc_path):
        kw["compression"] = "gzip"
    return pd.read_csv(junc_path, **kw)


def _count_data_rows_quick(junc_path: Path) -> int:
    opener = gzip.open if _is_gz(junc_path) else open
    n = 0
    with opener(junc_path, "rt") as fh:
        for _ in fh:
            n += 1
    return max(n - 1, 0)


def _parse_chunk_to_coo_triplets(
    chunk: pd.DataFrame,
    row_offset: int,
    threads: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[str]]:
    junc_ids = chunk.iloc[:, 0].astype(str).tolist()
    mat = chunk.iloc[:, 1:]  # strings
    n_cols = mat.shape[1]

    def work(j: int):
        v = pd.to_numeric(mat.iloc[:, j], errors="coerce").fillna(0).to_numpy(dtype=np.int32, copy=False)
        nz = np.nonzero(v)[0]
        if nz.size == 0:
            return (
                np.empty(0, dtype=np.int64),
                np.empty(0, dtype=np.int32),
                np.empty(0, dtype=np.int32),
            )
        return (
            (row_offset + nz).astype(np.int64),
            np.full(nz.size, j, dtype=np.int32),
            v[nz],
        )

    rows, cols, data = [], [], []
    if threads > 1 and n_cols > 1:
        with ThreadPoolExecutor(max_workers=threads) as ex:
            for r, c, d in ex.map(work, range(n_cols)):
                if r.size:
                    rows.append(r); cols.append(c); data.append(d)
    else:
        for j in range(n_cols):
            r, c, d = work(j)
            if r.size:
                rows.append(r); cols.append(c); data.append(d)

    if rows:
        R = np.concatenate(rows)
        C = np.concatenate(cols)
        D = np.concatenate(data)
    else:
        R = np.empty(0, dtype=np.int64)
        C = np.empty(0, dtype=np.int32)
        D = np.empty(0, dtype=np.int32)

    return R, C, D, junc_ids


def _load_sparse_from_juncmatrix(
    junc_path: Path,
    chunksize: int,
    threads: int,
    estimate_total: bool,
    show_progress: bool = True,
) -> tuple[sparse.csr_matrix, pd.Index, pd.Index]:
    barcodes = _read_header_barcodes(junc_path)
    n_cells = len(barcodes)

    total_rows = _count_data_rows_quick(junc_path) if estimate_total else None
    pbar = tqdm(
        total=total_rows,
        unit="rows",
        desc="Reading juncmatrix",
        smoothing=0.05,
        disable=not show_progress,
    )

    allR, allC, allD = [], [], []
    junc_ids: list[str] = []
    row_offset = 0

    for chunk in _stream_chunks(junc_path, chunksize):
        R, C, D, J = _parse_chunk_to_coo_triplets(chunk, row_offset, threads)
        if D.size:
            allR.append(R); allC.append(C); allD.append(D)
        junc_ids.extend(J)
        row_offset += len(J)
        pbar.update(len(J))

    pbar.close()

    if allR:
        R = np.concatenate(allR)
        C = np.concatenate(allC)
        D = np.concatenate(allD)
        coo = sparse.coo_matrix((D, (R, C)), shape=(row_offset, n_cells), dtype=np.int32)
    else:
        coo = sparse.coo_matrix((row_offset, n_cells), dtype=np.int32)

    X = coo.tocsr().T  # cells × junctions
    return X, pd.Index(barcodes, name="barcode"), pd.Index(junc_ids, name="junction")


# Robust attr parsing (regex) to avoid silent failures due to spacing/order
_re_gene_id = re.compile(r'gene_id "([^"]+)"')
_re_gene_name = re.compile(r'gene_name "([^"]+)"')

def _load_geneName_to_geneId_from_gtf(gtf: Path) -> dict[str, str]:
    """
    Build geneName -> geneId mapping from GTF(.gz).
    Also includes gene_id -> gene_id.
    Uses regex to be robust to attribute ordering/spacing.
    """
    opener = gzip.open if _is_gz(gtf) else open
    m: dict[str, str] = {}
    with opener(gtf, "rt") as fh:
        for line in fh:
            if not line or line[0] == "#":
                continue
            f = line.rstrip("\n").split("\t")
            if len(f) < 9 or f[2] != "gene":
                continue
            attrs = f[8]

            mi = _re_gene_id.search(attrs)
            if not mi:
                continue
            gene_id = mi.group(1)

            mn = _re_gene_name.search(attrs)
            gene_name = mn.group(1) if mn else None

            m[gene_id] = gene_id
            if gene_name:
                m[gene_name] = gene_id
    return m


def _infer_sample_name(junc_path: Path) -> str:
    name = junc_path.name
    m = re.match(r"^(?P<s>.+)_juncmatrix\.txt(\.gz)?$", name)
    if m:
        return m["s"]
    return junc_path.parent.name


def _store_complete_zarr(store: Path) -> bool:
    return store.exists() and (store / "X").exists() and (store / "obs").exists() and (store / "var").exists()


def _choose_chunks(n_cells: int, n_junc: int) -> tuple[int, int]:
    chunk_cells = min(8192, max(1024, n_cells))
    chunk_juncs = min(32768, max(4096, (n_junc // 64) or 4096))
    return chunk_cells, chunk_juncs


def juncmatrix_to_zarr(
    junc_path: Union[str, Path],
    out_zarr: Union[str, Path],
    *,
    sample: Optional[str] = None,
    gtf: Optional[Union[str, Path]] = None,
    threads: int = 4,
    chunksize: int = 20_000,
    estimate_total: bool = False,
    gene_progress: bool = False,
    cellmeta_path: Optional[Union[str, Path]] = None,
    barcode_prefix: Optional[str] = None,
    overwrite: bool = False,
    show_progress: bool = True,
) -> Path:
    """
    Convert one SiCeLoRe-style juncmatrix(.txt|.gz) to a sparse AnnData Zarr store.

    Output AnnData:
      - X: sparse CSR, shape (cells × junctions)
      - obs: barcode index (prefixed), includes 'sample' + optional cell metadata
      - var: index is junction id, contains ONLY 'geneId'
        where geneId is:
          - Ensembl gene_id if GTF mapping succeeds
          - otherwise the junction prefix (gene symbol) so it is NEVER NaN
    """
    junc_path = Path(junc_path)
    out_zarr = Path(out_zarr)

    if not junc_path.exists():
        raise FileNotFoundError(f"Missing juncmatrix: {junc_path}")

    if (not overwrite) and _store_complete_zarr(out_zarr):
        return out_zarr

    sample_name = sample or _infer_sample_name(junc_path)
    prefix = barcode_prefix if barcode_prefix is not None else sample_name

    X, barcodes, junc_ids = _load_sparse_from_juncmatrix(
        junc_path=junc_path,
        chunksize=int(chunksize),
        threads=int(threads),
        estimate_total=bool(estimate_total),
        show_progress=show_progress,
    )

    # obs
    obs_index = pd.Index([f"{prefix}#{b}" for b in barcodes], name="barcode")
    obs = pd.DataFrame(index=obs_index)
    obs["sample"] = sample_name

    if cellmeta_path is not None:
        cellmeta_path = Path(cellmeta_path)
        if cellmeta_path.exists():
            sep = "\t" if cellmeta_path.suffix.lower() in {".tsv"} else ","
            cb = pd.read_csv(cellmeta_path, sep=sep)
            bcol = "barcode" if "barcode" in cb.columns else cb.columns[0]
            cb = cb.rename(columns={bcol: "barcode"})
            cb["barcode"] = prefix + "#" + cb["barcode"].astype(str)
            obs = obs.join(cb.set_index("barcode"), how="left")

    # var (ONLY geneId)
    var = pd.DataFrame(index=junc_ids.astype(str))

    # Always derive a usable per-gene key from the junction prefix
    gene_prefix = var.index.to_series().astype(str).str.split(":", n=1, expand=True)[0]

    if gtf is not None:
        gtf = Path(gtf)
        if not gtf.exists():
            raise FileNotFoundError(f"GTF not found: {gtf}")

        name_to_id = _load_geneName_to_geneId_from_gtf(gtf)
        mapped = gene_prefix.map(name_to_id)

        # CRITICAL: never leave geneId NaN — fallback to the prefix
        var["geneId"] = mapped.where(mapped.notna(), gene_prefix)

        if gene_progress and show_progress:
            n_ok = int(mapped.notna().sum())
            tqdm.write(f"[geneId] mapped {n_ok}/{var.shape[0]} via GTF; fallback filled the rest")
    else:
        # No GTF: still guaranteed non-null geneId for slicing
        var["geneId"] = gene_prefix

    # (Optional) enforce non-null in case of pathological strings
    var["geneId"] = var["geneId"].astype(str)
    bad = (var["geneId"].isna()) | (var["geneId"].eq("nan")) | (var["geneId"].eq("None")) | (var["geneId"].eq(""))
    if bad.any():
        # last-resort fill from prefix
        var.loc[bad, "geneId"] = gene_prefix.loc[bad].astype(str)

    A = ad.AnnData(X=X, obs=obs, var=var)

    n_cells, n_junc = A.shape
    chunk_cells, chunk_juncs = _choose_chunks(n_cells, n_junc)

    out_zarr.parent.mkdir(parents=True, exist_ok=True)
    A.write_zarr(str(out_zarr), chunks=(chunk_cells, chunk_juncs))
    return out_zarr


def batch_juncmatrix_to_zarr(
    inputs: Union[str, Path, Sequence[Union[str, Path]]],
    outroot: Union[str, Path],
    *,
    gtf: Optional[Union[str, Path]] = None,
    threads: int = 4,
    chunksize: int = 20_000,
    estimate_total: bool = False,
    gene_progress: bool = False,
    overwrite: bool = False,
    show_progress: bool = True,
) -> list[Path]:
    outroot = Path(outroot)
    outroot.mkdir(parents=True, exist_ok=True)

    paths: list[Path] = []
    if isinstance(inputs, (str, Path)):
        p = Path(inputs)
        if p.exists() and p.is_dir():
            for fp in p.rglob("*_juncmatrix.txt*"):
                if fp.is_file():
                    paths.append(fp)
        else:
            for fp in glob.glob(str(inputs), recursive=True):
                fp2 = Path(fp)
                if fp2.is_file():
                    paths.append(fp2)
    else:
        for x in inputs:
            fp = Path(x)
            if fp.is_file():
                paths.append(fp)

    paths = sorted({pp.resolve() for pp in paths})
    if not paths:
        raise ValueError("No juncmatrix files found from inputs.")

    results: list[Path] = []
    it = tqdm(paths, desc="juncmatrix→zarr", unit="file", disable=not show_progress)

    for jpath in it:
        sample = _infer_sample_name(jpath)
        out = Path(outroot) / sample
        try:
            res = juncmatrix_to_zarr(
                junc_path=jpath,
                out_zarr=out,
                sample=sample,
                gtf=gtf,
                threads=threads,
                chunksize=chunksize,
                estimate_total=estimate_total,
                gene_progress=gene_progress,
                overwrite=overwrite,
                show_progress=show_progress,
            )
            results.append(res)
        except Exception as e:
            if show_progress:
                tqdm.write(f"[warn] failed {jpath}: {e}")
            else:
                print(f"[warn] failed {jpath}: {e}")

    return results


def merge_junction_zarrs(
    root_or_paths: Union[str, Path, Sequence[Union[str, Path]]],
    out_zarr: Union[str, Path],
    *,
    join: str = "outer",
    sample_key: str = "sample",
    ensure_unique_barcodes: bool = True,
    skip_hidden: bool = True,
    skip_names: Iterable[str] = ("ANALYSIS",),
    overwrite: bool = False,
    show_progress: bool = True,
) -> Path:
    out_zarr = Path(out_zarr)
    if (not overwrite) and _store_complete_zarr(out_zarr):
        return out_zarr

    stores: list[Path] = []
    if isinstance(root_or_paths, (str, Path)):
        p = Path(root_or_paths)
        if p.exists() and p.is_dir():
            for sub in sorted(p.iterdir()):
                if not sub.is_dir():
                    continue
                name = sub.name
                if skip_hidden and (name.startswith(".") or name.startswith("_")):
                    continue
                if name.upper() in {s.upper() for s in skip_names}:
                    continue
                stores.append(sub)
        else:
            for s in sorted(glob.glob(str(root_or_paths))):
                sp = Path(s)
                if sp.is_dir():
                    stores.append(sp)
    else:
        for x in root_or_paths:
            xp = Path(x)
            if xp.is_dir():
                stores.append(xp)

    if not stores:
        raise ValueError("No per-sample Zarr stores found to merge.")

    alist: list[ad.AnnData] = []
    keys: list[str] = []
    it = tqdm(stores, desc="Read sample zarr", unit="sample", disable=not show_progress)

    for store in it:
        sname = store.name
        A = ad.read_zarr(str(store))

        if ensure_unique_barcodes:
            if not all(str(b).startswith(sname + "#") for b in A.obs_names):
                A.obs_names = [f"{sname}#{b}" for b in A.obs_names]

        if sample_key not in A.obs:
            A.obs[sample_key] = sname

        if "geneId" not in A.var:
            # if a store is missing geneId entirely, derive from junction index as last resort
            gene_prefix = A.var_names.to_series().astype(str).str.split(":", n=1, expand=True)[0]
            A.var["geneId"] = gene_prefix

        # keep ONLY geneId in merged var
        A.var = A.var[["geneId"]].copy()

        alist.append(A)
        keys.append(sname)

    Aall = ad.concat(
        alist,
        axis=0,
        join=join,
        merge="first",
        label=sample_key,
        keys=keys,
        index_unique=None,
        fill_value=0,
    )

    if "geneId" not in Aall.var:
        Aall.var["geneId"] = Aall.var_names.to_series().astype(str).str.split(":", n=1, expand=True)[0]
    Aall.var = Aall.var[["geneId"]].copy()

    n_cells, n_junc = Aall.shape
    chunk_cells, chunk_juncs = _choose_chunks(n_cells, n_junc)

    out_zarr.parent.mkdir(parents=True, exist_ok=True)
    Aall.write_zarr(str(out_zarr), chunks=(chunk_cells, chunk_juncs))
    return out_zarr

# Junction Data Processing

> Utilities for processing SiCeLoRe-style junction matrices into AnnData Zarr format for scalable analysis.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| default_exp junctions

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()